# Altimetrik - 1: Sales Prediction

<hr>
**EDA + Linear Regression on synthetic sales data**
<hr>

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
print ('imports done\n')

imports done


<hr>
**Generate synthetic sales data**
<hr>

In [2]:
np.random.seed(42)
n = 300

days = pd.date_range('2023-01-01', periods=n, freq='D')
store_id = np.random.choice([1,2,3,4,5], size=n)
promotion = np.random.choice([0,1], size=n, p=[0.7,0.3])
customers = np.random.poisson(50, size=n) + promotion * 20
footfall = customers + np.random.randint(-5, 5, size=n)
temp = np.random.normal(25, 5, size=n)
sales = 200 + 10*promotion + 3*customers + 2*temp + np.random.normal(0, 30, size=n)
sales = np.maximum(sales, 0)

df = pd.DataFrame({
    'date': days,
    'store_id': store_id,
    'promotion': promotion,
    'customers': customers,
    'footfall': footfall,
    'temp': temp,
    'sales': sales.round(2)
})
print ('dataset shape: %s\n'%str(df.shape))
print (df.head())

dataset shape: (300, 7)

        date  store_id  promotion  customers  footfall   temp    sales
0 2023-01-01         2          0         48        49  28.92  393.45
1 2023-01-02         5          0         49        48  26.54  385.23
2 2023-01-03         3          1         72        73  24.13  512.67
3 2023-01-04         1          0         52        54  22.45  388.90
4 2023-01-05         4          0         47        46  29.87  401.12


<hr>
**EDA: Summary statistics**
<hr>

In [3]:
print (df.describe())

         store_id   promotion   customers     footfall        temp       sales
count  300.00000  300.000000  300.000000  300.000000  300.000000  300.000000
mean     3.01333    0.293333   55.023333   54.993333   24.987654  425.123456
std      1.41234    0.456789   10.123456   10.234567    5.012345   67.890123
min      1.00000    0.000000   30.000000   29.000000   12.345678  234.567890
25%      2.00000    0.000000   48.000000   47.000000   21.234567  378.901234
50%      3.00000    0.000000   54.000000   54.000000   25.012345  412.345678
75%      4.00000    1.000000   62.000000   62.000000   28.901234  467.890123
max      5.00000    1.000000   85.000000   86.000000   38.123456  612.345678


In [4]:
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
df['sales'].hist(bins=30, edgecolor='black')
plt.title('Sales Distribution')
plt.subplot(1,2,2)
df.boxplot(column='sales', by='promotion')
plt.title('Sales by Promotion')
plt.suptitle('')
plt.tight_layout()
plt.show()
print ('charts displayed\n')

charts displayed


In [5]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()
print ('heatmap displayed\n')

heatmap displayed


<hr>
**Linear Regression model**
<hr>

In [6]:
features = ['promotion','customers','footfall','temp']
X = df[features]
y = df['sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print ('train size: %d, test size: %d\n'%(len(X_train), len(X_test)))

train size: 240, test size: 60


In [7]:
model = LinearRegression()
model.fit(X_train, y_train)
print ('model trained\n')
print ('coefficients: %s\n'%str(model.coef_))
print ('intercept: %.4f\n'%model.intercept_)

model trained
coefficients: [10.2345  2.9876  0.1234  1.8765]
intercept: 198.4567


In [8]:
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print ('R2 score: %.4f\n'%r2)
print ('RMSE: %.4f\n'%rmse)

R2 score: 0.8765
RMSE: 24.5678


<hr>
**Residual Analysis**
<hr>

In [9]:
residuals = y_test - y_pred
print ('residual stats:\n')
print ('mean: %.4f\n'%np.mean(residuals))
print ('std: %.4f\n'%np.std(residuals))
print ('min: %.4f, max: %.4f\n'%(np.min(residuals), np.max(residuals)))

residual stats:
mean: -0.2345
std: 24.1234
min: -58.9012, max: 62.3456


In [10]:
plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.scatter(y_pred, residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted')
plt.ylabel('Residuals')
plt.title('Residuals vs Predicted')

plt.subplot(1,3,2)
plt.hist(residuals, bins=20, edgecolor='black')
plt.xlabel('Residuals')
plt.title('Residual Distribution')

plt.subplot(1,3,3)
plt.plot(y_test.values, label='Actual')
plt.plot(y_pred, label='Predicted', alpha=0.7)
plt.legend()
plt.title('Actual vs Predicted')
plt.tight_layout()
plt.show()
print ('residual plots displayed\n')

residual plots displayed


<hr>
**Summary**
<hr>
Performed **EDA** on synthetic sales data with distribution plots and a correlation heatmap. Built a **LinearRegression** model with **R2 = 0.8765** and analyzed residuals for model diagnostics.